In [ ]:
### preparing data for heatmap ###
import sys
import numpy as np
import matplotlib.pyplot as plt

from concurrent.futures import ProcessPoolExecutor, as_completed
import argparse
from tqdm import tqdm

import multivariate
import matplotlib

def pairwise_statistics(W, alpha):
    max_pairwise = np.nan
    ## Now we should consider two iterables
    ## Still save all the sampled max statistics
#     for iter_0 in range(W.shape[0]):
        ## slot for TS among all iterations
    T_ij = np.full(W.shape[2], np.nan)
    for iter_1 in range(W.shape[2]):
        TS = np.full(int(W.shape[1] * (W.shape[1] - 1) / 2), np.nan)
        sum = 0
        for iter_i in range(W.shape[1] - 1):
            for iter_j in range(iter_i + 1, W.shape[1]):
                TS[sum] = np.max(W[: , iter_i ,iter_1] - W[: , iter_j ,iter_1])
                sum += 1
        T_ij[iter_1] = np.max(TS)
    max_pairwise = np.quantile(T_ij, 1 - alpha / 2)
    return max_pairwise

def pairwise_statistics_step(W, alpha, idx):
    W = W[:,idx,:] 
    max_pairwise = np.full(W.shape[0], np.nan)
    ## Now we should consider two iterables
    ## Still save all the sampled max statistics
    for iter_0 in range(W.shape[0]):
        ## slot for TS among all iterations
        T_ij = np.full(W.shape[2], np.nan)
        for iter_1 in range(W.shape[2]):
            TS = np.full(int(W.shape[1] * (W.shape[1] - 1) / 2), np.nan)
            sum = 0
            for iter_i in range(W.shape[1] - 1):
                for iter_j in range(iter_i + 1, W.shape[1]):
                    TS[sum] = np.max(W[iter_0, : , iter_i ,iter_1] - W[iter_0, : , iter_j ,iter_1])
                    sum += 1
            T_ij[iter_1] = np.max(TS)
        max_pairwise[iter_0] = np.quantile(T_ij, 1 - alpha / 2)
    return max_pairwise

def test_statistics(est, h, dim, time_num, filt):
    if filt:
        T = generate_grid(0, 1, 3, time_num)
        mask = (T > 0.2) & (T < 0.8)
        # Use np.all() to filter rows where all conditions are true
        filtered_T = T[np.all(mask, axis=1)]
        indices = np.where(np.all(mask, axis=1))[0]
        est = est[indices,:]
        
    T = np.zeros((est.shape[1], est.shape[1]))
#     for iter_0 in range(est.shape[0]):
        ## slot for TS among all iterations
    for iter_i in range(est.shape[1]):
        for iter_j in range(est.shape[1]):
            T[iter_i, iter_j] = np.min(est[: , iter_i] - est[: , iter_j])
    return T


def test_statistics_step(est, idx):
    est = est[:,:,idx]
    T = np.zeros((est.shape[1], est.shape[1]))
#     for iter_0 in range(est.shape[0]):
        ## slot for TS among all iterations
    for iter_i in range(est.shape[1]):
        for iter_j in range(est.shape[1]):
            T[iter_0, iter_i, iter_j] = np.min(est[iter_0, : , iter_i] - est[iter_0, : , iter_j])
    return T

def global_test(T, quantile):
    ## for each iter
#     ratio_all = np.full(T.shape[0], np.nan)
#     for iter_0 in range(T.shape[0]):
    TS = np.full(int(T.shape[1] * (T.shape[1] - 1) / 2), False)
    q = quantile
    sum = 0
    for iter_i in range(T.shape[1] - 1):
                    for iter_j in range(iter_i + 1, T.shape[1]):
                        TS[sum] = (T[iter_i, iter_j] < q)
                        sum += 1
    ratio = np.all(TS)
#     ratio = np.sum(ratio_all)/ T.shape[0]
    return ratio

## Generate multivariate grids
def generate_grid(begin, end, dim, time_num):
    # Compute the number of points per dimension
    points_per_dim = int(round(time_num ** (1/dim)))

    # Generate a linearly spaced grid for each dimension
    grid1d = np.linspace(begin, end, points_per_dim)
    
    # Generate meshgrid for all dimensions
    mesh = np.meshgrid(*([grid1d]*dim))
    
    # Flatten and combine to form the final grid
    grid_points = np.vstack(map(np.ravel, mesh)).T
    
    # If the number of points exceeds time_num, truncate the array
    if len(grid_points) > time_num:
        grid_points = grid_points[:time_num]

    return grid_points

In [ ]:
##### test dim = 1 ########

import numpy as np
import os
# Load the .npz file
# n = 20
# L_values=[50, 100, 200, 300] #[50,100,200]
# h_values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# k_values=[0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.09]

n = 50
L_values=[100,200] #[50,100,200]
h_values=[0.3] #[0.2,0.5,0.8]
k_values = [0.01]
p_values = [0.5,0.8]

dim = 1
iterr = 1
m = 0.1
rep =5
pp = 100.0
for Lij in L_values:
    for h in h_values:
        for k in k_values:
            for p in p_values:
                qq = np.full(rep,np.nan)
                tt = np.full(rep,np.nan)
                Tmax = np.full((rep,n,n),np.nan)
                Tu = np.full(rep,np.nan)
                Tl = np.full(rep,np.nan)
                for seed in range(rep):
    #                 n20_Lij50_p0.2_h0.5_lambda1e-05_dim3_m0.1_k0.01_seed9.npz
                    	# n20_Lij100_p0.8_h0.3_lambda1e-05_dim1_m0.1_k0.01+0.1_num100.0_seed4_c10.npz
                    filename = f'results/n={n}/n{n}_Lij{Lij}_p{p}_h{h}_lambda1e-05_dim{dim}_m{m}_k{k}+0.1_num{pp}_seed{seed+1}_c10.npz'

                    if os.path.exists(filename):
                        data = np.load(filename)
    #                     print(f'Loaded {filename}')
                        est = data['estimates']
                        ground_truth = data['ground_truth']
                        GMB_W = data['GMB_W']

                        T = test_statistics(est,h,dim,200,True)
                        quantile = pairwise_statistics(GMB_W, 0.9)  # Assuming you want the 50th percentile
    #                     print(quantile)
    #                     print(global_test(T, quantile))
    #                     upper_triangle = np.triu(matrix, k=1)

                        Tmax[seed] = T
                        Tu[seed] = np.round(np.max(np.triu(T,k=1)),2)
                        Tl[seed] = np.round(np.max(np.tril(T,k=1)),2)
                        qq[seed] = np.round(quantile,2)
                        tt[seed] = global_test(T, quantile)
    #                 else:
    #                     print(f"File not found: {filename}")
                tt = tt[~np.isnan(tt)]
                print(f'n{n}_Lij{Lij}_p{p}_h{h}_lambda1e-05_dim3_m{m}_k{k}')
                print(qq)
                print(Tu)
                print(Tl)
                print(np.sum(tt)/len(tt))
    #             print(Tmax)


In [ ]:
##### collect T ########
import os
# Load the .npz file
# n = 20
# L_values=[50, 100, 200, 300] #[50,100,200]
# h_values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# k_values=[0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.09]

n = 20
L_values=[50,100] #[50,100,200]
h_values=[0.22,0.23] #[0.2,0.5,0.8]
 #[0.2,0.5,0.8]
# h_values= [ 0.22]
k_values = [10.0]
c_values = [2.0]

dim = 3
iterr = 1
m = 0.1
rep = 10
pp = 400.0
for Lij in L_values:
    for h in h_values:
        for k in k_values:
            for c in c_values:
                qq = np.full(rep,np.nan)
                tt = np.full(rep,np.nan)
                Tmax = np.full((rep,n,n),np.nan)
                for seed in range(rep):
    #                 n20_Lij50_p0.2_h0.4_lambda1e-05_dim3_m0.1_k1.0_num400.0_seed9_c4.npz
                    filename = f'PATH/TO/_c4.npz'
                    if os.path.exists(filename):

                        data = np.load(filename)
                        est = data['estimates']
                        ground_truth = data['ground_truth']
                        GMB_W = data['GMB_W']

                        T = test_statistics(est,h,dim,200,True)
                        quantile = pairwise_statistics(GMB_W, 0.95)  # Assuming you want the 50th percentile
                        Tmax[seed] = T
    #                     print(quantile)
                        qq[seed] = np.round(quantile,2)
                        tt[seed] = global_test(T, quantile)
    #                 else:
    #                     print('Y')

                tt = tt[~np.isnan(tt)]
                print(f'n{n}_Lij{Lij}_p0.2_h{h}_lambda1e-05_dim3_m{m}_k{k}+{c}')
                print(qq)
                print(np.round(np.nanmax(Tmax, axis=(1, 2)),2) )
                print(np.sum(tt)/len(tt))
#             print(Tmax)


In [ ]:
n = 50
Lij = 100
seed = 0
h = 0.05

qq = np.full(rep,np.nan)
tt = np.full(rep,np.nan)
Tmax = np.full((rep,n,n),np.nan)

for seed in range(rep):
    filename = f'PATH/TO/_c4.npz'
    if os.path.exists(filename):

        data = np.load(filename)
        est = data['estimates']
        ground_truth = data['ground_truth']
        GMB_W = data['GMB_W']

        T = test_statistics(est,h,dim,200,True)
        quantile = pairwise_statistics(GMB_W, 0.9)  # Assuming you want the 50th percentile
        Tmax[seed] = T
#                     print(quantile)
        qq[seed] = np.round(quantile,2)
        tt[seed] = global_test(T, quantile)
#                 else:
#                     print('Y')

tt = tt[~np.isnan(tt)]
print(f'n{n}_Lij{Lij}_p0.2_h{h}_lambda1e-05_dim3_m{m}_k{k}')
print(qq)
print(np.round(np.nanmax(Tmax, axis=(1, 2)),2) )
print(np.sum(tt)/len(tt))
#             print(Tmax)


In [ ]:
# import pandas as pd
rep = 50
Imax = np.full((rep,n,n),np.nan)
for seed in range(rep):
    Imax[seed] = (Tmax[seed]>qq[seed])

# import pandas as pd
n = 20
Lij = 100
h = 0.23
m = 0.1
k = 1.0
c= 1
p = 0.2
rep = 50
for seed in range(0,rep):
    df = np.array(Imax[seed,:,:])
    np.savetxt(f'results/out/n{n}_Lij{Lij}_p{p}_h{h}_dim3_m{m}_k{k}+{c}_alpha0.9_c7_seed{seed}', df, fmt='%d', delimiter=' ')


df0 = np.sum(Imax, axis=0)/rep
np.savetxt(f'results/out/n{n}_Lij{Lij}_p{p}_h{h}_dim3_m{m}_k{k}+{c}_alpha0.9_c7_ratio', df0, fmt='%0.2f')   